# Multi-Modal AI: Vision, Audio & Beyond

**Track:** LLMOps  
**Toolchain:** GPT-4V · Gemini Vision · CLIP · Whisper  
**Objective:** Learn how to build applications that understand images, audio, and documents — not just text.

---

## 🧠 Why Multi-Modal in 2026?

The first 95% of LLM applications used text-only. But in 2026, the most impactful applications combine multiple modalities:

| Application | Modalities | Business Value |
|------------|-----------|---------------|
| Document processing | Image + Text | Extract data from invoices, receipts, forms |
| Medical imaging | Image + Text | Analyze X-rays, describe findings |
| Customer support | Audio + Text | Transcribe calls, analyze sentiment |
| Visual QA | Image + Text | "What's wrong with this dashboard screenshot?" |
| Video analysis | Video + Text + Audio | Meeting summaries, content moderation |

### 🤔 What Does "Multi-Modal" Actually Mean?

A **modality** is a type of data:
- **Text** → words, sentences, documents
- **Image** → photos, screenshots, diagrams
- **Audio** → speech, music, sound effects
- **Video** → sequences of images + audio

A **multi-modal model** can process MULTIPLE types at once. GPT-4V can see images AND read text. Gemini can process video AND audio.

### Multi-Modal Models (2026)

| Model | Text | Image | Audio | Video | Best For |
|-------|------|-------|-------|-------|---------|
| **GPT-4o** | ✅ | ✅ | ✅ | ❌ | General purpose, image understanding |
| **Gemini 2.0** | ✅ | ✅ | ✅ | ✅ | Video, longest context, multimodal |
| **Claude 3.5** | ✅ | ✅ | ❌ | ❌ | Document analysis, coding |
| **Llama 3.2 Vision** | ✅ | ✅ | ❌ | ❌ | Open-source image understanding |
| **Whisper v3** | ❌ | ❌ | ✅ | ❌ | Speech-to-text (open source) |
| **CLIP** | ✅ | ✅ | ❌ | ❌ | Image-text embeddings (open source) |

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** The future isn't text; it's multimodal spatial intelligence. Seniors utilize Vision API encoders and Audio transcription models to build AI systems that can see dashboards, read physical invoices, and hear meetings.

**Mental Model:** Multimodal embeddings (like CLIP) map entirely different senses (a photo of a cat, the sound of a meow, the literal word 'cat') into the exact same point in mathematical space, enabling cross-modal search.

**Common Junior Pitfall:** Sending massive 4K resolution images to a Vision API blindly, resulting in $0.15 API calls and slow latency per image. Seniors downscale images or use 'low-detail' modes for basic classification tasks.

---


In [ ]:
# Cell 1 — Install
!pip install -q pillow numpy pydantic

## 1 · Vision: Image Understanding in Production

### 🤔 What Can Vision Models Do?

| Task | Example | API Approach |
|------|---------|-------------|
| **Describe** | "What's in this image?" | Send image + simple prompt |
| **Extract** | "Read the text from this receipt" | Send image + extraction prompt |
| **Analyze** | "Any defects in this product photo?" | Send image + analysis prompt |
| **Compare** | "How do these two designs differ?" | Send multiple images + comparison prompt |
| **Classify** | "Is this image NSFW?" | Send image + classification schema |

### Image Input Formats

| Method | When to Use | Token Cost |
|--------|-----------|------------|
| **URL** | Image hosted online | Same |
| **Base64** | Local file, dynamic generation | Same |
| **Low resolution** | Classification, simple Q&A | ~85 tokens |
| **High resolution** | OCR, detail extraction | ~765 tokens per 512x512 tile |

In [ ]:
# Cell 2 — Vision API request building
import base64, json
from pydantic import BaseModel, Field
from typing import List, Optional

# Structure for vision API request (OpenAI format)
def build_vision_request(image_path_or_url, prompt, detail='auto'):
    """Build a multi-modal API request.
    
    This is the ACTUAL format used by OpenAI, Anthropic, and Gemini APIs.
    """
    # Determine if URL or local file
    if image_path_or_url.startswith('http'):
        image_content = {"type": "image_url", "image_url": {"url": image_path_or_url, "detail": detail}}
    else:
        # In real code: open file, base64 encode
        fake_b64 = base64.b64encode(b'fake_image_data').decode()
        image_content = {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{fake_b64}", "detail": detail}}
    
    request = {
        "model": "gpt-4o",
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                image_content,
            ]
        }],
        "max_tokens": 1000,
    }
    return request

# Production use cases
class InvoiceExtraction(BaseModel):
    vendor_name: str
    invoice_number: str
    date: str
    total_amount: float
    line_items: List[dict]
    currency: str = "USD"

use_cases = [
    {
        'name': 'Invoice Processing',
        'prompt': 'Extract all fields from this invoice image. Return JSON with: vendor_name, invoice_number, date, total_amount, line_items, currency.',
        'image': 'https://example.com/invoice.png',
        'detail': 'high',  # Need OCR-level detail
    },
    {
        'name': 'Product Quality Check',
        'prompt': 'Inspect this product photo for defects. Return: has_defect (bool), defect_type, severity (low/medium/high), location.',
        'image': 'https://example.com/product.jpg',
        'detail': 'high',
    },
    {
        'name': 'Dashboard Screenshot Analysis',
        'prompt': 'What metrics are shown in this dashboard? Are any in a warning/critical state?',
        'image': 'https://example.com/dashboard.png',
        'detail': 'low',  # Don't need pixel-level detail
    },
]

print('🖼️ Vision API Use Cases')
print('=' * 60)
for uc in use_cases:
    req = build_vision_request(uc['image'], uc['prompt'], uc['detail'])
    content_types = [c['type'] for c in req['messages'][0]['content']]
    est_tokens = 85 if uc['detail'] == 'low' else 765
    print(f'\n  📋 {uc["name"]}')
    print(f'     Detail: {uc["detail"]} (~{est_tokens} image tokens)')
    print(f'     Content: {content_types}')
    print(f'     Prompt: "{uc["prompt"][:60]}..."')

print(f'\n💡 Use "low" detail for classification (cheaper).')
print(f'   Use "high" detail for OCR and extraction (more accurate).')

---

## 2 · Audio: Speech-to-Text & Beyond

### 🤔 When Do You Need Audio Processing?

| Use Case | Pipeline | Key Model |
|----------|---------|----------|
| Meeting transcription | Audio → Whisper → Text | Whisper v3 |
| Voice assistants | Audio → Whisper → LLM → TTS → Audio | Full pipeline |
| Call center analytics | Audio → Whisper → Sentiment analysis | Whisper + LLM |
| Podcast chapters | Audio → Whisper → Summarize → Chapters | Whisper + LLM |

### Tools for Audio

| Tool | Direction | Quality | Cost |
|------|-----------|---------|------|
| **Whisper v3** (OpenAI) | Speech → Text | Excellent | Free (local) / $0.006/min (API) |
| **Deepgram** | Speech → Text | Excellent | $0.0043/min |
| **ElevenLabs** | Text → Speech | Excellent | $0.30/1K chars |
| **Bark** (Suno) | Text → Speech | Good | Free (open source) |

In [ ]:
# Cell 3 — Audio processing pipeline simulation
import json

class AudioPipeline:
    """Simulates a production audio processing pipeline."""
    
    def transcribe(self, audio_duration_sec):
        """Stage 1: Speech to text (Whisper)."""
        words_per_second = 2.5  # average speaking rate
        word_count = int(audio_duration_sec * words_per_second)
        cost = audio_duration_sec / 60 * 0.006  # $0.006/minute
        transcript = f'[Simulated transcript of {word_count} words from {audio_duration_sec}s audio]'
        return {'transcript': transcript, 'words': word_count, 'cost': cost}
    
    def analyze(self, transcript):
        """Stage 2: LLM analysis of transcript."""
        return {
            'summary': 'Discussion about Q4 results and product roadmap.',
            'action_items': ['Review budget proposal', 'Schedule follow-up'],
            'sentiment': 'positive',
            'speakers': 3,
            'cost': 0.02  # LLM API cost
        }
    
    def process_meeting(self, duration_minutes):
        """Full pipeline: audio → transcript → analysis."""
        duration_sec = duration_minutes * 60
        t = self.transcribe(duration_sec)
        a = self.analyze(t['transcript'])
        return {
            'duration': f'{duration_minutes} min',
            'words': t['words'],
            'summary': a['summary'],
            'action_items': a['action_items'],
            'total_cost': f"${t['cost'] + a['cost']:.3f}",
        }

pipeline = AudioPipeline()

print('🎙️ Audio Processing Pipeline')
print('=' * 60)
for duration in [5, 30, 60]:
    result = pipeline.process_meeting(duration)
    print(f'\n  📞 {result["duration"]} meeting ({result["words"]} words)')
    print(f'     Summary: {result["summary"]}')
    print(f'     Actions: {result["action_items"]}')
    print(f'     Cost: {result["total_cost"]}')

print(f'\n💡 A 1-hour meeting costs ~$0.38 to fully transcribe and analyze.')
print(f'   This replaces manual note-taking and saves hours of work.')

---

## 3 · Multi-Modal Embeddings (CLIP)

### 🤔 What is CLIP?

CLIP (by OpenAI) embeds images AND text into the SAME vector space. This means you can:
- Search images with text queries ("find me sunset photos")
- Search text with image queries (reverse image search)
- Classify images without training (zero-shot)

```
Text: "a cat sleeping"  → vector [0.3, -0.1, 0.8, ...]
Image: [photo of cat]   → vector [0.3, -0.1, 0.7, ...]  ← Similar!
Image: [photo of car]   → vector [-0.5, 0.4, -0.2, ...] ← Different!
```

### Multi-Modal RAG

Traditional RAG searches text with text. **Multi-modal RAG** can:
- Search documents using image queries
- Include image chunks alongside text chunks in context
- Answer questions about diagrams, charts, and screenshots

In [ ]:
# Cell 4 — Multi-modal embedding and search
import numpy as np

class MultiModalEmbedder:
    """Simulates CLIP-style multi-modal embeddings."""
    
    def __init__(self, dim=512):
        self.dim = dim
    
    def embed_text(self, text):
        np.random.seed(hash(text) % 2**32)
        vec = np.random.randn(self.dim)
        return vec / np.linalg.norm(vec)
    
    def embed_image(self, image_description):
        # CLIP puts images and text in the SAME space
        np.random.seed(hash(image_description) % 2**32)
        vec = np.random.randn(self.dim) + 0.1  # Slight offset to simulate real differences
        return vec / np.linalg.norm(vec)

embedder = MultiModalEmbedder()

# Build a multi-modal index
items = [
    {'type': 'text',  'content': 'Q4 revenue grew 15% to $58.1M'},
    {'type': 'image', 'content': 'revenue chart showing upward trend'},
    {'type': 'text',  'content': 'Engineering headcount increased to 67'},
    {'type': 'image', 'content': 'org chart of engineering department'},
    {'type': 'text',  'content': 'Customer satisfaction score reached 8.1'},
    {'type': 'image', 'content': 'NPS survey results dashboard screenshot'},
]

# Embed all items
for item in items:
    if item['type'] == 'text':
        item['vector'] = embedder.embed_text(item['content'])
    else:
        item['vector'] = embedder.embed_image(item['content'])

# Search with a text query
query = 'financial results and revenue'
query_vec = embedder.embed_text(query)

# Find nearest items (text OR images)
results = []
for item in items:
    sim = np.dot(query_vec, item['vector'])
    results.append({**item, 'similarity': sim})

results.sort(key=lambda x: x['similarity'], reverse=True)

print(f'🔍 Multi-Modal Search: "{query}"')
print('=' * 60)
for r in results[:4]:
    icon = '📝' if r['type'] == 'text' else '🖼️'
    print(f'  {icon} [{r["type"]:>5}] sim={r["similarity"]:+.3f} | {r["content"]}')

print(f'\n💡 Multi-modal search returns BOTH text and images in one query.')
print(f'   The revenue chart appears alongside the revenue text!')

---

## 🎯 Summary & Key Takeaways

| Modality | Key Model | Common Use Case |
|----------|----------|----------------|
| Vision | GPT-4o, Gemini 2.0 | Document processing, quality inspection |
| Audio | Whisper v3 | Meeting transcription, voice assistants |
| Multi-modal embeddings | CLIP | Cross-modal search, zero-shot classification |
| Multi-modal RAG | CLIP + LLM | Answer questions about documents with images |

### 🧠 When to Go Multi-Modal

1. **Document has images/charts** → Vision + RAG
2. **User speaks instead of types** → Whisper + LLM
3. **Need to search images with text** → CLIP embeddings
4. **Video understanding** → Gemini 2.0 (best video support)

**Next →** Agentic Orchestration & Memory Architectures